# 09 — Platform Engineering: Batch, Stream, and Promotion Gates

**Objectives**
- Compare Lambda (batch + speed) with Kappa (stream-only) pipelines.
- Encode model promotion as executable policy, not tribal knowledge.
- Simulate a CI gate that blocks unqualified models.


In [ ]:
from collections import defaultdict

events = [
    {"minute": 0, "value": 5}, {"minute": 0, "value": 3},
    {"minute": 1, "value": 8}, {"minute": 2, "value": 2},
]

# Stream view: incremental fold, visible immediately
stream = defaultdict(int)
for e in events:
    stream[e["minute"]] += e["value"]
print("stream_aggregates:", dict(sorted(stream.items())))

# Batch view: reprocess everything at the end of the window
batch = defaultdict(int)
for e in events:
    batch[e["minute"]] += e["value"]
print("batch_matches_stream:", dict(batch) == dict(stream))


In [ ]:
# Model promotion policy
def promotion_gate(metrics: dict) -> dict:
    rules = {
        "accuracy >= 0.90": metrics["accuracy"] >= 0.90,
        "auc >= 0.85": metrics["auc"] >= 0.85,
        "latency_p95_ms <= 150": metrics["latency_p95_ms"] <= 150,
        "no_regression_vs_champion": metrics["accuracy"] >= metrics["champion_accuracy"],
    }
    return {"pass": all(rules.values()), "rules": rules}

candidates = {
    "model_a": {"accuracy": 0.92, "auc": 0.88, "latency_p95_ms": 120, "champion_accuracy": 0.91},
    "model_b": {"accuracy": 0.94, "auc": 0.90, "latency_p95_ms": 240, "champion_accuracy": 0.91},
}
decisions = {name: promotion_gate(m) for name, m in candidates.items()}
for name, decision in decisions.items():
    print(name, "promote" if decision["pass"] else "block", decision["rules"])
print("promoted_count:", sum(1 for d in decisions.values() if d["pass"]))


In [ ]:
# CI gate: all stages must pass before an artifact is deployable
stages = {"unit": True, "integration": True, "performance": False, "security": True}
failed = [name for name, ok in stages.items() if not ok]
print("deploy_allowed:", not failed)
print("failed_stages:", failed)


**Decision guide**
- Kappa when replaying the log is cheap and one codebase is worth more than
  separate batch correctness.
- Promotion policy belongs in version control next to the model config so the
  same rules apply in CI and production.
